In [1]:
import csv
import os

def txt_to_csv(txt_file, csv_file, delimiter="\t"):
    """
    Convert a TXT file to CSV format.
    
    :param txt_file: Path to the input TXT file
    :param csv_file: Path to the output CSV file
    :param delimiter: Character separating values in TXT (default: tab)
    """
    if not os.path.exists(txt_file):
        raise FileNotFoundError(f"File not found: {txt_file}")

    try:
        with open(txt_file, "r", encoding="utf-8") as infile, \
             open(csv_file, "w", newline="", encoding="utf-8") as outfile:
            reader = csv.reader(infile, delimiter=delimiter)
            writer = csv.writer(outfile)
            for row in reader:
                writer.writerow(row)
        print(f"✅ Conversion complete: {csv_file}")
    except Exception as e:
        print(f"❌ Error: {e}")

# If you use the CSV file I provided in the repo, you can just execute this cell
# But if you have your own CSV, or if you changed mine's name, make sure you also update this line of the code
txt_to_csv("tickers_master_list.txt", "tickers_master_list.csv", delimiter=",")

✅ Conversion complete: tickers_master_list.csv


In [2]:
import pandas as pd
import yfinance as yf
import time

# =========================
# 1. LOAD TICKERS
# =========================
def load_tickers(csv_file):
    try:
        df = pd.read_csv(csv_file, delimiter=";", header=0)
        if "Symbol" not in df.columns:
            raise ValueError("CSV must contain a 'Symbol' column")
        
        tickers = df["Symbol"].dropna().unique().tolist()
        print(f"✅ Loaded {len(tickers)} tickers")
        return tickers

    except Exception as e:
        raise RuntimeError(f"Error loading CSV: {e}")


# =========================
# 2. VALIDATE TICKERS (FAST)
# =========================
def validate_tickers(tickers, period="5d"):
    print("⏳ Batch download...")

    data = yf.download(
        tickers,
        period=period,
        group_by="ticker",
        threads=True,
        auto_adjust=False,
        progress=False
    )

    valid = []
    retry = []

    # =========================
    # PASS 1: Batch validation
    # =========================
    for t in tickers:
        try:
            if isinstance(data.columns, pd.MultiIndex) and t in data.columns.levels[0]:
                df = data[t]
                if not df.empty and df["Close"].notna().any():
                    valid.append(t)
                else:
                    retry.append(t)
            else:
                retry.append(t)
        except Exception:
            retry.append(t)

    print(f"🔁 Retrying {len(retry)} uncertain tickers individually...")

    # =========================
    # PASS 2: Individual retry
    # =========================
    invalid = []

    for t in retry:
        try:
            df = yf.Ticker(t).history(period="5d")
            if not df.empty and df["Close"].notna().any():
                valid.append(t)
            else:
                invalid.append(t)
        except Exception:
            invalid.append(t)

    return valid, invalid

# =========================
# 3. MAIN EXECUTION
# =========================
if __name__ == "__main__":
    start = time.time()

    csv_file = "tickers_master_list.csv"

    tickers = load_tickers(csv_file)
    valid, invalid = validate_tickers(tickers)

    print("\n=========================")
    print(f"✅ Valid tickers: {len(valid)}")
    print(f"❌ Invalid tickers: {len(invalid)}")

    if invalid:
        print("\nInvalid list:")
        print(invalid)

    print("=========================")
    print(f"⏱ Execution time: {round(time.time() - start, 2)}s")

✅ Loaded 300 tickers
⏳ Batch download...
🔁 Retrying 0 uncertain tickers individually...

✅ Valid tickers: 300
❌ Invalid tickers: 0
⏱ Execution time: 12.25s
